# DeepEval Hallucination Evaluation

The purpose of this notebook is to run test cases of hallucination evaluations on datasets.

## Setup

In [ ]:
pip install --upgrade deepeval

In [12]:
from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np
import anthropic
import sys
import io
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.metrics import HallucinationMetric
from deepeval.test_case import LLMTestCase
from deepeval import evaluate

In [4]:
# Load environment files and store API keys
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_KEY")
gemini_key = os.getenv("GEMINI_KEY")
openai_key = os.getenv("OPENAI_KEY")

## Data

In [5]:
# Read in dialogue data
file_path = "../../data/Hallucination/dialogue_data.json"
dialogue_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/general_data.json"
general_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/qa_data.json"
qa_data = pd.read_json(file_path, lines=True)

In [ ]:
# Generate a random choice (True for right_answer, False for hallucinated_answer)
choice = np.random.rand(len(qa_data)) < 0.5

# Assign the chosen answer
qa_data["selected_answer"] = np.where(choice, qa_data["right_answer"], qa_data["hallucinated_answer"])

# Add a column indicating the source of the answer
qa_data["hallucinated_flag"] = np.where(choice, 0, 1)

## LLM Connection

In [ ]:
class Claude(DeepEvalBaseLLM):
    """Class to implement Claude model for DeepEval"""
    def __init__(self, model, api_key):
        self.model = model
        self.api_key = api_key
        self.client = anthropic.Client(api_key=self.api_key)

    def load_model(self):
        return self.model

    def generate(self, prompt, max_tokens = 10000):
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        return response.content[0].text

    async def a_generate(self, prompt, max_tokens = 10000):
        
        model = self.load_model()
        
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        return response.content[0].text

    def get_model_name(self):
        return "Claude Model"

In [ ]:
# Initialize the Claude model with appropriate settings
claude_model = "claude-3-7-sonnet-20250219"

# Create Claude model
claude_instance = Claude(model=claude_model, api_key=anthropic_key)

# Test functionality
print(claude_instance.generate("Hello, Claude"))

Hello! It's nice to meet you. How can I assist you today?


## Evaluation Test Cases

In [ ]:
# Function to query the LLM with hallucination-aware prompt
def evaluate_hallucination(context, question):
    
    '''
    Evaluates the hallucination of an AI assistant to a question, given context.

    The function generates a prompt for the AI, asking it to answer a question using only the 
    provided context. It then measures whether the answer contains hallucinations, i.e., 
    information not supported by the context. The function returns the generated response, 
    a hallucination score, and the reasoning behind the score.

    Parameters:
    context (str): The text or information that the AI assistant should use to answer the question.
    question (str): The question to be answered by the AI assistant.

    Returns:
    tuple: A tuple containing three elements:
        - response (str): The generated response from the AI assistant.
        - hallucination_score (float): A score indicating the score of hallucination in the response.
        - reasoning (str): The reasoning or explanation for the hallucination score.
    '''
    
    # prompt = f"""
    # You are an AI assistant that strictly answers questions based on the given context.
    # If the answer is not in the context, say "I don’t know based on the provided information."
    # Do NOT make up facts, infer missing details, or add extra information beyond the context.

    # Context:
    # {context}

    # Question:
    # {question}
    
    # Answer:
    # """

    # Generate response to the prompt
    response = ''# claude_instance.generate(prompt)
    
    # Create DeepEval test case
    test_case = LLMTestCase(
        input=f"\n\nHuman:{question}\n\nAssistant:",
        actual_output=f"\n\n{response}\n\nAssistant:",
        context=[f"\n\n{context}\n\nAssistant:"]
    )

    # Create hallucination evaluation metric
    metric = HallucinationMetric(model = claude_instance, threshold=0.5)

    # Avoid unnecessary printing from DeepEval functions
    original_stdout = sys.stdout
    sys.stdout = io.StringIO()
    
    # Run evaluation using test case and metric
    evaluation = evaluate([test_case], [metric])

    # Avoid unnecessary printing from DeepEval functions
    sys.stdout = original_stdout
    
    # Store hallucination score
    hallucination_score = evaluation.test_results[0].metrics_data[0].score
    
    # Store reasoning for score
    reasoning = evaluation.test_results[0].metrics_data[0].reason
    
    # Return outputs
    return hallucination_score, reasoning
    # return response, hallucination_score, reasoning

In [ ]:
# Initialize list to hold results
results = []

# Iterate through entire dataset
for index, row in qa_data.iterrows():
    # Get results
    score, reasoning = evaluate_hallucination(row.knowledge, row.question, row.selected_answer)
    # response, score, reasoning = evaluate_hallucination(row.knowledge, row.question, row.response)
    
    # Store results
    results.append({
        "question": row.question,
        "context": row.knowledge,
        "answer": row.selected_answer, # response,
        "hallucination_score": score,
        "reasoning": reasoning,
        "hallucination_flag": row.hallucination_flag
    })

In [ ]:
# Initialize list to hold results
# results = []
# 
# # Iterate through dataset (starting with 10 records)
# for i in range(0,10):
# 
#     # Define context and question
#     context = qa_data.knowledge[i]
#     question = qa_data.question[i]
#     
#     # Get results
#     response, score, reasoning = evaluate_hallucination(context, question)
#     
#     # Store results
#     results.append({
#         "question": question,
#         "context": context,
#         "llm_answer": response,
#         "hallucination_score": score,
#         "reasoning": reasoning
#         
#     })

In [ ]:
# Convert to data frame
results_df = pd.DataFrame(results)
results_df.head(10)

,question,context,llm_answer,hallucination_score,reasoning
0,Which magazine was started first Arthur's Maga...,Arthur's Magazine (1844–1846) was an American ...,"Based on the provided information, Arthur's Ma...",0.0,The score is 0.00 because the actual output al...
1,The Oberoi family is part of a hotel company t...,The Oberoi family is an Indian family that is ...,"Based on the provided context, The Oberoi Grou...",0.0,The score is 0.00 because the actual output fu...
2,Musician and satirist Allie Goertz wrote a son...,"Allison Beth ""Allie"" Goertz (born March 2, 199...","Based on the provided context, Matt Groening n...",0.0,The score is 0.00 because the actual output fu...
3,What nationality was James Henry Miller's wife?,"Margaret ""Peggy"" Seeger (born June 17, 1935) i...","Based on the provided context, James Henry Mil...",0.0,The score is 0.00 because the actual output al...
4,Cadmium Chloride is slightly soluble in this c...,It is a hygroscopic solid that is highly solu...,"Based on the provided context, I don't know th...",0.0,The score is 0.00 because the actual output ac...
5,Which tennis player won more Grand Slam titles...,"Jonathan Stark (born April 3, 1971) is a forme...",I don't know based on the provided information...,1.0,The score is 1.00 because the actual output co...
6,Which genus of moth in the world's seventh-lar...,Indogrammodes is a genus of moths of the Cramb...,"Based on the provided context, the genus of mo...",0.0,The score is 0.00 because the actual output al...
7,Who was once considered the best kick boxer in...,Fighters from around world on the roster incl...,"Based on the provided context, Badr Hari was o...",0.0,The score is 0.00 because the actual output fu...
8,"The Dutch-Belgian television series that ""Hous...",House of Anubis is a mystery television series...,"Based on the provided context, the Dutch-Belgi...",0.0,The score is 0.00 because the actual output fu...
9,What is the length of the track where the 2013...,The 2013 Liqui Moly Bathurst 12 Hour was an en...,"The Mount Panorama Circuit, where the 2013 Liq...",0.0,The score is 0.00 because the actual output pe...
